# Silver Layer

## Silver Pipeline Initialization

### Imports and Logger Setup
Initializes required libraries and configures logging.

### Pipeline Start
Logs the start of the Silver layer processing.

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_carts_pipeline")

logger.info("Silver pipeline started")

INFO:silver_carts_pipeline:Silver pipeline started


## Silver Data Transformation

### Logger Setup
Initializes logging for transformation process.

### Data Read
Loads data from Bronze table in Unity Catalog.

### Preview
Displays the dataset for inspection.

In [0]:
from pyspark.sql.functions import *
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_transformation")

logger.info("Silver transformation started")

bronze_df = spark.table("ecom_catalog.bronze.carts_bronze")



INFO:silver_transformation:Silver transformation started


In [0]:

display(bronze_df.limit(5))

## Data Flattening

### Explode Nested Data
Flattens nested carts array into individual records.

### Output
Displays flattened carts data.

In [0]:
carts_flat_df = bronze_df.select(
    explode("carts").alias("cart")
)



INFO:py4j.clientserver:Received command c on object id p0


In [0]:
display(carts_flat_df.limit(5))

## Data Transformation

### Select and Rename Columns
Extracts required fields from cart data and renames columns.

### Explode Products
Flattens nested products within each cart for detailed analysis.

In [0]:
final_silver_df = carts_flat_df.select(
    col("cart.id").alias("cart_id"),
    col("cart.userId").alias("user_id"),
    col("cart.total").alias("cart_total"),
    col("cart.discountedTotal").alias("cart_discounted_total"),
    col("cart.totalProducts").alias("total_products"),
    col("cart.totalQuantity").alias("total_quantity"),

    explode(col("cart.products")).alias("product")
)


cart level data is same but item level is unique


## Final Data Preparation

### Column Selection
Selects required cart and product fields for analysis.

### Data Structuring
Creates a structured dataset with flattened cart and product details.

### Output
Displays the transformed Silver dataset.

In [0]:
final_silver_df = final_silver_df.select(
    "cart_id",
    "user_id",
    "cart_total",
    "cart_discounted_total",
    "total_products",
    "total_quantity",

    col("product.id").alias("product_id"),
    col("product.title").alias("product_name"),
    col("product.price").alias("price"),
    col("product.quantity").alias("quantity"),
    col("product.total").alias("product_total"),
    col("product.discountedTotal").alias("product_discounted_total"),
    col("product.discountPercentage").alias("discount_percentage"),
    col("product.thumbnail").alias("thumbnail")
)

display(final_silver_df.limit(20))

cart_id,user_id,cart_total,cart_discounted_total,total_products,total_quantity,product_id,product_name,price,quantity,product_total,product_discounted_total,discount_percentage,thumbnail
1,33,103774.85,89686.65,4,15,168,Charger SXT RWD,32999.99,3,98999.97,85743.87,13.39,https://cdn.dummyjson.com/products/images/vehicle/Charger%20SXT%20RWD/thumbnail.png
1,33,103774.85,89686.65,4,15,78,Apple MacBook Pro 14 Inch Space Grey,1999.99,2,3999.98,3259.18,18.52,https://cdn.dummyjson.com/products/images/laptops/Apple%20MacBook%20Pro%2014%20Inch%20Space%20Grey/thumbnail.png
1,33,103774.85,89686.65,4,15,183,Green Oval Earring,24.99,5,124.94999999999999,117.1,6.28,https://cdn.dummyjson.com/products/images/womens-jewellery/Green%20Oval%20Earring/thumbnail.png
1,33,103774.85,89686.65,4,15,100,Apple Airpods,129.99,5,649.95,566.5,12.84,https://cdn.dummyjson.com/products/images/mobile-accessories/Apple%20Airpods/thumbnail.png
2,142,4794.8,4288.95,5,20,144,Cricket Helmet,44.99,4,179.96,159.32,11.47,https://cdn.dummyjson.com/products/images/sports-accessories/Cricket%20Helmet/thumbnail.png
2,142,4794.8,4288.95,5,20,124,iPhone X,899.99,4,3599.96,3310.88,8.03,https://cdn.dummyjson.com/products/images/smartphones/iPhone%20X/thumbnail.png
2,142,4794.8,4288.95,5,20,148,Golf Ball,9.99,4,39.96,35.47,11.24,https://cdn.dummyjson.com/products/images/sports-accessories/Golf%20Ball/thumbnail.png
2,142,4794.8,4288.95,5,20,122,iPhone 6,299.99,3,899.97,723.22,19.64,https://cdn.dummyjson.com/products/images/smartphones/iPhone%206/thumbnail.png
2,142,4794.8,4288.95,5,20,110,Selfie Lamp with iPhone,14.99,5,74.95,60.06,19.87,https://cdn.dummyjson.com/products/images/mobile-accessories/Selfie%20Lamp%20with%20iPhone/thumbnail.png
3,108,16775.87,14144.3,5,13,98,Rolex Submariner Watch,13999.99,1,13999.99,11710.99,16.35,https://cdn.dummyjson.com/products/images/mens-watches/Rolex%20Submariner%20Watch/thumbnail.png


## Data Normalization

### Carts Table
Stores cart-level information such as user, totals, and quantities.

### Cart Items Table
Stores product-level details for each cart, enabling detailed analysis.

### Purpose
Separates data into structured tables to reduce redundancy and improve query performance.

## Cart Table

## Cart-Level Dataset

### Column Selection
Extracts cart-level fields from flattened data.

### Remove Duplicates
Ensures unique cart records.

### Output
Displays cleaned cart dataset.

In [0]:
cart_df = carts_flat_df.select(
    col("cart.id").alias("cart_id"),
    col("cart.userId").alias("user_id"),
    col("cart.total").alias("cart_total"),
    col("cart.discountedTotal").alias("cart_discounted_total"),
    col("cart.totalProducts").alias("total_products"),
    col("cart.totalQuantity").alias("total_quantity")
).dropDuplicates()

display(cart_df.limit(10))

INFO:py4j.clientserver:Received command c on object id p0


cart_id,user_id,cart_total,cart_discounted_total,total_products,total_quantity
20,90,3534.78,2936.58,6,22
1,33,103774.85,89686.65,4,15
22,140,222.88,205.63,4,12
29,170,3862.43,3488.44,5,17
17,141,192.87,173.9,4,13
25,118,412.98,353.55,2,2
30,177,128249.07,118740.76,4,13
7,86,145651.89,128504.41,3,11
13,41,793.07,750.8,5,13
14,94,586.82,500.25,6,18


Cart Items Table

## Cart Items Dataset

### Explode Products
Extracts product details from each cart.

### Column Selection
Selects and renames product-level fields.

### Output
Displays cart-wise product dataset.

In [0]:
cart_items_df = carts_flat_df.select(
    col("cart.id").alias("cart_id"),
    explode(col("cart.products")).alias("product")
).select(
    col("cart_id"),

    col("product.id").alias("product_id"),
    col("product.title").alias("product_name"),
    col("product.price").alias("price"),
    col("product.quantity").alias("quantity"),
    col("product.total").alias("product_total"),
    col("product.discountedTotal").alias("product_discounted_total"),
    col("product.discountPercentage").alias("discount_percentage"),
    col("product.thumbnail").alias("thumbnail")
)

display(cart_items_df.limit(20))

cart_id,product_id,product_name,price,quantity,product_total,product_discounted_total,discount_percentage,thumbnail
1,168,Charger SXT RWD,32999.99,3,98999.97,85743.87,13.39,https://cdn.dummyjson.com/products/images/vehicle/Charger%20SXT%20RWD/thumbnail.png
1,78,Apple MacBook Pro 14 Inch Space Grey,1999.99,2,3999.98,3259.18,18.52,https://cdn.dummyjson.com/products/images/laptops/Apple%20MacBook%20Pro%2014%20Inch%20Space%20Grey/thumbnail.png
1,183,Green Oval Earring,24.99,5,124.94999999999999,117.1,6.28,https://cdn.dummyjson.com/products/images/womens-jewellery/Green%20Oval%20Earring/thumbnail.png
1,100,Apple Airpods,129.99,5,649.95,566.5,12.84,https://cdn.dummyjson.com/products/images/mobile-accessories/Apple%20Airpods/thumbnail.png
2,144,Cricket Helmet,44.99,4,179.96,159.32,11.47,https://cdn.dummyjson.com/products/images/sports-accessories/Cricket%20Helmet/thumbnail.png
2,124,iPhone X,899.99,4,3599.96,3310.88,8.03,https://cdn.dummyjson.com/products/images/smartphones/iPhone%20X/thumbnail.png
2,148,Golf Ball,9.99,4,39.96,35.47,11.24,https://cdn.dummyjson.com/products/images/sports-accessories/Golf%20Ball/thumbnail.png
2,122,iPhone 6,299.99,3,899.97,723.22,19.64,https://cdn.dummyjson.com/products/images/smartphones/iPhone%206/thumbnail.png
2,110,Selfie Lamp with iPhone,14.99,5,74.95,60.06,19.87,https://cdn.dummyjson.com/products/images/mobile-accessories/Selfie%20Lamp%20with%20iPhone/thumbnail.png
3,98,Rolex Submariner Watch,13999.99,1,13999.99,11710.99,16.35,https://cdn.dummyjson.com/products/images/mens-watches/Rolex%20Submariner%20Watch/thumbnail.png


## Product Table


## Product Data Transformation

### Logger Setup
Initializes logging for product transformation process.

### Data Read
Loads products data from Bronze layer.

### Explode Nested Data
Flattens products array into individual records.

### Data Flattening
Extracts and structures fields including pricing, status, and identifiers.

### Complex Fields Handling
Flattens nested structures and retains arrays without exploding.

### Output
Displays the transformed product dataset.

In [0]:
from pyspark.sql.functions import *
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("product_silver")

logger.info("Product transformation started")


df = spark.table("ecom_catalog.bronze.products_bronze")


df_exploded = df.select(
    explode(col("products")).alias("p")
)


product_df = df_exploded.select(

    
    col("p.id").alias("product_id"),
    col("p.title").alias("title"),
    col("p.description").alias("description"),
    col("p.brand").alias("brand"),
    col("p.category").alias("category"),

    
    col("p.price").alias("price"),
    col("p.discountPercentage").alias("discount_percentage"),
    col("p.rating").alias("rating"),
    col("p.stock").alias("stock"),
    col("p.minimumOrderQuantity").alias("min_order_qty"),
    col("p.weight").alias("weight"),

    col("p.availabilityStatus").alias("availability_status"),
    col("p.returnPolicy").alias("return_policy"),
    col("p.shippingInformation").alias("shipping_info"),
    col("p.warrantyInformation").alias("warranty_info"),

   
    col("p.sku").alias("sku"),
    col("p.thumbnail").alias("thumbnail"),

    
    col("p.dimensions.depth").alias("depth"),
    col("p.dimensions.height").alias("height"),
    col("p.dimensions.width").alias("width"),

    col("p.meta.barcode").alias("barcode"),
    col("p.meta.createdAt").alias("created_at"),
    col("p.meta.updatedAt").alias("updated_at"),
    col("p.meta.qrCode").alias("qr_code"),

    
    col("p.images").alias("images"),
    col("p.tags").alias("tags")
)

product_df = product_df.select("*")

display(product_df.limit(20))

INFO:product_silver:Product transformation started


product_id,title,description,brand,category,price,discount_percentage,rating,stock,min_order_qty,weight,availability_status,return_policy,shipping_info,warranty_info,sku,thumbnail,depth,height,width,barcode,created_at,updated_at,qr_code,images,tags
1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,Essence,beauty,9.99,10.48,2.56,99,48,4,In Stock,No return policy,Ships in 3-5 business days,1 week warranty,BEA-ESS-ESS-001,https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,22.99,13.08,15.14,5784719087687,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),"List(beauty, mascara)"
2,Eyeshadow Palette with Mirror,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.",Glamour Beauty,beauty,19.99,18.19,2.86,34,20,9,In Stock,7 days return policy,Ships in 2 weeks,1 year warranty,BEA-GLA-EYE-002,https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,27.67,22.47,9.26,9170275171413,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),"List(beauty, eyeshadow)"
3,Powder Canister,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.",Velvet Touch,beauty,14.99,9.84,4.64,89,22,8,In Stock,No return policy,Ships in 1-2 business days,3 months warranty,BEA-VEL-POW-003,https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,20.59,27.93,29.27,8418883906837,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),"List(beauty, face powder)"
4,Red Lipstick,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.",Chic Cosmetics,beauty,12.99,12.16,4.36,91,40,1,In Stock,7 days return policy,Ships in 1 week,3 year warranty,BEA-CHI-LIP-004,https://cdn.dummyjson.com/product-images/beauty/red-lipstick/thumbnail.webp,22.17,28.38,18.11,9467746727219,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/red-lipstick/1.webp),"List(beauty, lipstick)"
5,Red Nail Polish,"The Red Nail Polish offers a rich and glossy red hue for vibrant and polished nails. With a quick-drying formula, it provides a salon-quality finish at home.",Nail Couture,beauty,8.99,11.44,4.32,79,22,8,In Stock,No return policy,Ships overnight,1 month warranty,BEA-NAI-NAI-005,https://cdn.dummyjson.com/product-images/beauty/red-nail-polish/thumbnail.webp,29.84,16.48,21.63,4063010628104,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/red-nail-polish/1.webp),"List(beauty, nail polish)"
6,Calvin Klein CK One,"CK One by Calvin Klein is a classic unisex fragrance, known for its fresh and clean scent. It's a versatile fragrance suitable for everyday wear.",Calvin Klein,fragrances,49.99,1.89,4.37,29,9,7,In Stock,90 days return policy,Ships overnight,1 week warranty,FRA-CAL-CAL-006,https://cdn.dummyjson.com/product-images/fragrances/calvin-klein-ck-one/thumbnail.webp,20.72,27.76,29.36,2451534060749,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,"List(https://cdn.dummyjson

# Transformation

## Data Cleaning

### Handle Missing Values
Fills null values in product, cart, and cart items datasets.

### Output
Displays cleaned datasets.

In [0]:
product_df = product_df.fillna({
    "brand": "Unknown",
    "category": "Unknown"
})

cart_df = cart_df.fillna(0)
cart_items_df = cart_items_df.fillna(0)

display(product_df.limit(20))
display(cart_df.limit(20))
display(cart_items_df.limit(20))

product_id,title,description,brand,category,price,discount_percentage,rating,stock,min_order_qty,weight,availability_status,return_policy,shipping_info,warranty_info,sku,thumbnail,depth,height,width,barcode,created_at,updated_at,qr_code,images,tags
1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,Essence,beauty,9.99,10.48,2.56,99,48,4,In Stock,No return policy,Ships in 3-5 business days,1 week warranty,BEA-ESS-ESS-001,https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,22.99,13.08,15.14,5784719087687,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),"List(beauty, mascara)"
2,Eyeshadow Palette with Mirror,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.",Glamour Beauty,beauty,19.99,18.19,2.86,34,20,9,In Stock,7 days return policy,Ships in 2 weeks,1 year warranty,BEA-GLA-EYE-002,https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,27.67,22.47,9.26,9170275171413,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),"List(beauty, eyeshadow)"
3,Powder Canister,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.",Velvet Touch,beauty,14.99,9.84,4.64,89,22,8,In Stock,No return policy,Ships in 1-2 business days,3 months warranty,BEA-VEL-POW-003,https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,20.59,27.93,29.27,8418883906837,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),"List(beauty, face powder)"
4,Red Lipstick,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.",Chic Cosmetics,beauty,12.99,12.16,4.36,91,40,1,In Stock,7 days return policy,Ships in 1 week,3 year warranty,BEA-CHI-LIP-004,https://cdn.dummyjson.com/product-images/beauty/red-lipstick/thumbnail.webp,22.17,28.38,18.11,9467746727219,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/red-lipstick/1.webp),"List(beauty, lipstick)"
5,Red Nail Polish,"The Red Nail Polish offers a rich and glossy red hue for vibrant and polished nails. With a quick-drying formula, it provides a salon-quality finish at home.",Nail Couture,beauty,8.99,11.44,4.32,79,22,8,In Stock,No return policy,Ships overnight,1 month warranty,BEA-NAI-NAI-005,https://cdn.dummyjson.com/product-images/beauty/red-nail-polish/thumbnail.webp,29.84,16.48,21.63,4063010628104,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/beauty/red-nail-polish/1.webp),"List(beauty, nail polish)"
6,Calvin Klein CK One,"CK One by Calvin Klein is a classic unisex fragrance, known for its fresh and clean scent. It's a versatile fragrance suitable for everyday wear.",Calvin Klein,fragrances,49.99,1.89,4.37,29,9,7,In Stock,90 days return policy,Ships overnight,1 week warranty,FRA-CAL-CAL-006,https://cdn.dummyjson.com/product-images/fragrances/calvin-klein-ck-one/thumbnail.webp,20.72,27.76,29.36,2451534060749,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,"List(https://cdn.dummyjson

cart_id,user_id,cart_total,cart_discounted_total,total_products,total_quantity
20,90,3534.78,2936.58,6,22
1,33,103774.85,89686.65,4,15
22,140,222.88,205.63,4,12
29,170,3862.43,3488.44,5,17
17,141,192.87,173.9,4,13
25,118,412.98,353.55,2,2
30,177,128249.07,118740.76,4,13
7,86,145651.89,128504.41,3,11
13,41,793.07,750.8,5,13
14,94,586.82,500.25,6,18


cart_id,product_id,product_name,price,quantity,product_total,product_discounted_total,discount_percentage,thumbnail
1,168,Charger SXT RWD,32999.99,3,98999.97,85743.87,13.39,https://cdn.dummyjson.com/products/images/vehicle/Charger%20SXT%20RWD/thumbnail.png
1,78,Apple MacBook Pro 14 Inch Space Grey,1999.99,2,3999.98,3259.18,18.52,https://cdn.dummyjson.com/products/images/laptops/Apple%20MacBook%20Pro%2014%20Inch%20Space%20Grey/thumbnail.png
1,183,Green Oval Earring,24.99,5,124.94999999999999,117.1,6.28,https://cdn.dummyjson.com/products/images/womens-jewellery/Green%20Oval%20Earring/thumbnail.png
1,100,Apple Airpods,129.99,5,649.95,566.5,12.84,https://cdn.dummyjson.com/products/images/mobile-accessories/Apple%20Airpods/thumbnail.png
2,144,Cricket Helmet,44.99,4,179.96,159.32,11.47,https://cdn.dummyjson.com/products/images/sports-accessories/Cricket%20Helmet/thumbnail.png
2,124,iPhone X,899.99,4,3599.96,3310.88,8.03,https://cdn.dummyjson.com/products/images/smartphones/iPhone%20X/thumbnail.png
2,148,Golf Ball,9.99,4,39.96,35.47,11.24,https://cdn.dummyjson.com/products/images/sports-accessories/Golf%20Ball/thumbnail.png
2,122,iPhone 6,299.99,3,899.97,723.22,19.64,https://cdn.dummyjson.com/products/images/smartphones/iPhone%206/thumbnail.png
2,110,Selfie Lamp with iPhone,14.99,5,74.95,60.06,19.87,https://cdn.dummyjson.com/products/images/mobile-accessories/Selfie%20Lamp%20with%20iPhone/thumbnail.png
3,98,Rolex Submariner Watch,13999.99,1,13999.99,11710.99,16.35,https://cdn.dummyjson.com/products/images/mens-watches/Rolex%20Submariner%20Watch/thumbnail.png


Remove duplicates

## Remove Duplicates

### Deduplication
Removes duplicate records based on primary keys.

### Output
Displays deduplicated datasets.

In [0]:
product_df = product_df.dropDuplicates(["product_id"])
cart_df = cart_df.dropDuplicates(["cart_id"])
cart_items_df = cart_items_df.dropDuplicates(["cart_id", "product_id"])

display(product_df.limit(20))
display(cart_df.limit(20))
display(cart_items_df.limit(20))

product_id,title,description,brand,category,price,discount_percentage,rating,stock,min_order_qty,weight,availability_status,return_policy,shipping_info,warranty_info,sku,thumbnail,depth,height,width,barcode,created_at,updated_at,qr_code,images,tags
28,Ice Cream,"Creamy and delicious ice cream, available in various flavors for a delightful treat.",Unknown,groceries,5.49,8.69,3.39,27,42,1,In Stock,No return policy,Ships in 2 weeks,1 month warranty,GRO-BRD-CRE-028,https://cdn.dummyjson.com/product-images/groceries/ice-cream/thumbnail.webp,24.2,15.07,14.83,0788954559076,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,"List(https://cdn.dummyjson.com/product-images/groceries/ice-cream/1.webp, https://cdn.dummyjson.com/product-images/groceries/ice-cream/2.webp, https://cdn.dummyjson.com/product-images/groceries/ice-cream/3.webp, https://cdn.dummyjson.com/product-images/groceries/ice-cream/4.webp)",List(desserts)
29,Juice,"Refreshing fruit juice, packed with vitamins and great for staying hydrated.",Unknown,groceries,3.99,12.06,3.94,50,25,1,In Stock,No return policy,Ships in 1 week,6 months warranty,GRO-BRD-JUI-029,https://cdn.dummyjson.com/product-images/groceries/juice/thumbnail.webp,28.02,21.46,18.56,6936112580956,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/groceries/juice/1.webp),List(beverages)
30,Kiwi,"Nutrient-rich kiwi, perfect for snacking or adding a tropical twist to your dishes.",Unknown,groceries,2.49,15.22,4.93,99,30,5,In Stock,7 days return policy,Ships overnight,6 months warranty,GRO-BRD-KIW-030,https://cdn.dummyjson.com/product-images/groceries/kiwi/thumbnail.webp,17.13,18.67,19.4,2530169917252,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,List(https://cdn.dummyjson.com/product-images/groceries/kiwi/1.webp),List(fruits)
9,Dolce Shine Eau de,"Dolce Shine by Dolce & Gabbana is a vibrant and fruity fragrance, featuring notes of mango, jasmine, and blonde woods. It's a joyful and youthful scent.",Dolce & Gabbana,fragrances,69.99,0.62,3.96,4,2,6,Low Stock,7 days return policy,Ships in 1 month,3 year warranty,FRA-DOL-DOL-009,https://cdn.dummyjson.com/product-images/fragrances/dolce-shine-eau-de/thumbnail.webp,18.3,29.88,27.28,3023868210708,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,"List(https://cdn.dummyjson.com/product-images/fragrances/dolce-shine-eau-de/1.webp, https://cdn.dummyjson.com/product-images/fragrances/dolce-shine-eau-de/2.webp, https://cdn.dummyjson.com/product-images/fragrances/dolce-shine-eau-de/3.webp)","List(fragrances, perfumes)"
11,Annibale Colombo Bed,"The Annibale Colombo Bed is a luxurious and elegant bed frame, crafted with high-quality materials for a comfortable and stylish bedroom.",Annibale Colombo,furniture,1899.99,8.57,4.77,88,1,10,In Stock,No return policy,Ships in 1 month,1 year warranty,FUR-ANN-ANN-011,https://cdn.dummyjson.com/product-images/furniture/annibale-colombo-bed/thumbnail.webp,17.28,25.36,28.16,3610757456581,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,https://cdn.dummyjson.com/public/qr-code.png,"List(https://cdn.dummyjson.com/product-images/furniture/annibale-colombo-bed/1.webp, https://cdn.dummyjson.com/product-images/furniture/annibale-colombo-bed/2.webp, https://cdn.dummyjson.com/product-images/furniture/annibale-colombo-bed/3.webp)","List(furniture, beds)"
14,Knoll Saarinen Executive Conference Chair,"The Knoll Saarinen Executive Conference Chair is a modern and ergonomic chair, perfect for your office or conference room with its timeless design.",Knoll,furniture,499.99,2.01,4.88,26,5,10,In Stock,60 days return policy,Ships overnight,2 year warranty,FUR-KNO-KNO-014,https://cdn.dummyjson.com/product-images/furniture/knoll-saarinen-executive-conference-chair/thumbnail.webp,5.62,7.5,13.81,8919386859966,2025-04-30T09:41:02.053Z,2025

cart_id,user_id,cart_total,cart_discounted_total,total_products,total_quantity
28,147,209.91,186.09,2,9
29,170,3862.43,3488.44,5,17
30,177,128249.07,118740.76,4,13
9,194,65529.87,63336.57,4,13
11,172,568.36,476.76,6,14
14,94,586.82,500.25,6,18
17,141,192.87,173.9,4,13
3,108,16775.87,14144.3,5,13
4,87,456.83,443.23,4,17
19,59,933.87,794.78,4,13


cart_id,product_id,product_name,price,quantity,product_total,product_discounted_total,discount_percentage,thumbnail
2,124,iPhone X,899.99,4,3599.96,3310.88,8.03,https://cdn.dummyjson.com/products/images/smartphones/iPhone%20X/thumbnail.png
6,112,TV Studio Camera Pedestal,499.99,3,1499.97,1264.62,15.69,https://cdn.dummyjson.com/products/images/mobile-accessories/TV%20Studio%20Camera%20Pedestal/thumbnail.png
9,178,Corset Leather With Skirt,89.99,2,179.98,157.32,12.59,https://cdn.dummyjson.com/products/images/womens-dresses/Corset%20Leather%20With%20Skirt/thumbnail.png
15,111,Selfie Stick Monopod,12.99,3,38.97,34.69,10.98,https://cdn.dummyjson.com/products/images/mobile-accessories/Selfie%20Stick%20Monopod/thumbnail.png
17,74,Spoon,4.99,4,19.96,19.41,2.78,https://cdn.dummyjson.com/products/images/kitchen-accessories/Spoon/thumbnail.png
20,90,Puma Future Rider Trainers,89.99,5,449.95,383.81,14.7,https://cdn.dummyjson.com/products/images/mens-shoes/Puma%20Future%20Rider%20Trainers/thumbnail.png
27,83,Blue & Black Check Shirt,29.99,5,149.95,136.81,8.76,https://cdn.dummyjson.com/products/images/mens-shirts/Blue%20&%20Black%20Check%20Shirt/thumbnail.png
29,153,Volleyball,11.99,5,59.95,50.33,16.05,https://cdn.dummyjson.com/products/images/sports-accessories/Volleyball/thumbnail.png
30,46,Plant Pot,14.99,4,59.96,49.38,17.65,https://cdn.dummyjson.com/products/images/home-decoration/Plant%20Pot/thumbnail.png
1,168,Charger SXT RWD,32999.99,3,98999.97,85743.87,13.39,https://cdn.dummyjson.com/products/images/vehicle/Charger%20SXT%20RWD/thumbnail.png


## Data Type Conversion

### Timestamp Conversion
Converts created_at field to timestamp format.

In [0]:
from pyspark.sql.functions import to_timestamp

product_df = product_df.withColumn("created_at", to_timestamp("created_at"))

## Standardization

### Text Normalization
Converts category and brand fields to lowercase for consistency.

In [0]:
from pyspark.sql.functions import lower

product_df = product_df.withColumn("category", lower("category"))
product_df = product_df.withColumn("brand", lower("brand"))

## Feature Engineering

### Revenue Calculation
Creates a new column to calculate revenue using price and quantity.

### Output
Displays dataset with revenue field.

In [0]:
cart_items_df = cart_items_df.withColumn(
    "revenue",
    col("price") * col("quantity")
)
display(cart_items_df.limit(20))

INFO:py4j.clientserver:Received command c on object id p0


cart_id,product_id,product_name,price,quantity,product_total,product_discounted_total,discount_percentage,thumbnail,revenue
2,124,iPhone X,899.99,4,3599.96,3310.88,8.03,https://cdn.dummyjson.com/products/images/smartphones/iPhone%20X/thumbnail.png,3599.96
6,112,TV Studio Camera Pedestal,499.99,3,1499.97,1264.62,15.69,https://cdn.dummyjson.com/products/images/mobile-accessories/TV%20Studio%20Camera%20Pedestal/thumbnail.png,1499.97
9,178,Corset Leather With Skirt,89.99,2,179.98,157.32,12.59,https://cdn.dummyjson.com/products/images/womens-dresses/Corset%20Leather%20With%20Skirt/thumbnail.png,179.98
15,111,Selfie Stick Monopod,12.99,3,38.97,34.69,10.98,https://cdn.dummyjson.com/products/images/mobile-accessories/Selfie%20Stick%20Monopod/thumbnail.png,38.97
17,74,Spoon,4.99,4,19.96,19.41,2.78,https://cdn.dummyjson.com/products/images/kitchen-accessories/Spoon/thumbnail.png,19.96
20,90,Puma Future Rider Trainers,89.99,5,449.95,383.81,14.7,https://cdn.dummyjson.com/products/images/mens-shoes/Puma%20Future%20Rider%20Trainers/thumbnail.png,449.95
27,83,Blue & Black Check Shirt,29.99,5,149.95,136.81,8.76,https://cdn.dummyjson.com/products/images/mens-shirts/Blue%20&%20Black%20Check%20Shirt/thumbnail.png,149.95
29,153,Volleyball,11.99,5,59.95,50.33,16.05,https://cdn.dummyjson.com/products/images/sports-accessories/Volleyball/thumbnail.png,59.95
30,46,Plant Pot,14.99,4,59.96,49.38,17.65,https://cdn.dummyjson.com/products/images/home-decoration/Plant%20Pot/thumbnail.png,59.96
1,168,Charger SXT RWD,32999.99,3,98999.97,85743.87,13.39,https://cdn.dummyjson.com/products/images/vehicle/Charger%20SXT%20RWD/thumbnail.png,98999.97


## Feature Engineering

### Discount Calculation
Computes discount value as the difference between total and discounted total.

### Output
Displays dataset with discount value field.

In [0]:
cart_items_df = cart_items_df.withColumn(
    "discount_value",
    col("product_total") - col("product_discounted_total")
)
display(cart_items_df.limit(20))

cart_id,product_id,product_name,price,quantity,product_total,product_discounted_total,discount_percentage,thumbnail,revenue,discount_value
2,124,iPhone X,899.99,4,3599.96,3310.88,8.03,https://cdn.dummyjson.com/products/images/smartphones/iPhone%20X/thumbnail.png,3599.96,289.0799999999999
6,112,TV Studio Camera Pedestal,499.99,3,1499.97,1264.62,15.69,https://cdn.dummyjson.com/products/images/mobile-accessories/TV%20Studio%20Camera%20Pedestal/thumbnail.png,1499.97,235.35000000000014
9,178,Corset Leather With Skirt,89.99,2,179.98,157.32,12.59,https://cdn.dummyjson.com/products/images/womens-dresses/Corset%20Leather%20With%20Skirt/thumbnail.png,179.98,22.659999999999997
15,111,Selfie Stick Monopod,12.99,3,38.97,34.69,10.98,https://cdn.dummyjson.com/products/images/mobile-accessories/Selfie%20Stick%20Monopod/thumbnail.png,38.97,4.280000000000001
17,74,Spoon,4.99,4,19.96,19.41,2.78,https://cdn.dummyjson.com/products/images/kitchen-accessories/Spoon/thumbnail.png,19.96,0.5500000000000007
20,90,Puma Future Rider Trainers,89.99,5,449.95,383.81,14.7,https://cdn.dummyjson.com/products/images/mens-shoes/Puma%20Future%20Rider%20Trainers/thumbnail.png,449.95,66.13999999999999
27,83,Blue & Black Check Shirt,29.99,5,149.95,136.81,8.76,https://cdn.dummyjson.com/products/images/mens-shirts/Blue%20&%20Black%20Check%20Shirt/thumbnail.png,149.95,13.139999999999986
29,153,Volleyball,11.99,5,59.95,50.33,16.05,https://cdn.dummyjson.com/products/images/sports-accessories/Volleyball/thumbnail.png,59.95,9.620000000000005
30,46,Plant Pot,14.99,4,59.96,49.38,17.65,https://cdn.dummyjson.com/products/images/home-decoration/Plant%20Pot/thumbnail.png,59.96,10.579999999999998
1,168,Charger SXT RWD,32999.99,3,98999.97,85743.87,13.39,https://cdn.dummyjson.com/products/images/vehicle/Charger%20SXT%20RWD/thumbnail.png,98999.97,13256.100000000006


## Joins

### Join with Products
Joins cart items data with product details.

### Added Fields
Includes category and brand information.

### Purpose
Enhances dataset for better analysis.

In [0]:
products_df = spark.table("ecom_catalog.silver.products")

cart_items_enriched_df = cart_items_df.join(
    products_df.select("product_id", "category", "brand"),
    on="product_id",
    how="left"
)

cart_items_enriched_df = cart_items_df.join(
    product_master_df,
    on="product_id",
    how="left"
)
display(cart_items_enriched_df)

## Aggregation

### Category-Level Analysis
Calculates total revenue and quantity for each category.

### Purpose
Provides insights into category-wise sales performance.

In [0]:
category_sales_df = cart_items_enriched_df.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

In [0]:
top_products_df = cart_items_enriched_df.groupBy("product_id", "product_name").agg(
    sum("quantity").alias("total_sold")
)

## Data Partitioning

### Partition Column
Creates a partition column based on category.

### Purpose
Improves query performance and data organization.

In [0]:
partitioned_df = cart_items_enriched_df.withColumn(
    "category_partition",
    col("category")
)

# Incremental load


In [0]:
bronze_df = spark.table("ecom_catalog.bronze.carts_bronze")

silver_df = spark.table("ecom_catalog.silver.carts")

INFO:py4j.clientserver:Received command c on object id p0


## Incremental Processing

### New Data Identification
Filters records present in Bronze but not in Silver.

### Method
Uses left anti join to capture only new data.

### Purpose
Supports incremental data loading.

In [0]:
new_data = bronze_df.join(
    silver_df,
    on="cart_id",
    how="left_anti"
)

## Incremental Load Logic

### Table Check
Verifies if Silver table already exists.

### Conditional Processing
If exists, loads only new records using left anti join.

### Initial Load
If not exists, processes full dataset.

### Purpose
Ensures efficient incremental data loading.

In [0]:
if spark.catalog.tableExists("ecom_catalog.silver.carts"):
    silver_df = spark.table("ecom_catalog.silver.carts")

    new_data = bronze_df.join(
        silver_df,
        on="cart_id",
        how="left_anti"
    )
else:
    new_data = bronze_df

# Write data 

## Write to Silver Tables

### Delta Storage
Writes transformed data into Silver layer using Delta format.

### Tables Created
Includes products, carts, and cart items tables.

### Partitioning
Cart items table is partitioned by category.

### Purpose
Stores cleaned and structured data for downstream processing.

In [0]:
product_df.write.mode("overwrite").format("delta").saveAsTable("ecom_catalog.silver.products")

cart_df.write.mode("overwrite").format("delta").saveAsTable("ecom_catalog.silver.carts")

cart_items_enriched_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .partitionBy("category") \
    .saveAsTable("ecom_catalog.silver.cart_items")

## Product Master Data

### Source Combination
Combines product data from products and cart items datasets.

### Data Enrichment
Fills missing category and brand with default values.

### Deduplication
Removes duplicate records based on product_id.

### Purpose
Creates a unified product master dataset.

In [0]:
cart_products_df = cart_items_df.select(
    col("product_id"),
    col("product_name").alias("title"),
    col("thumbnail")
).dropDuplicates()

product_master_df = product_df.select(
    "product_id", "title", "category", "brand"
).unionByName(
    cart_products_df.select(
        "product_id", "title"
    ).withColumn("category", lit("unknown"))
     .withColumn("brand", lit("unknown"))
).dropDuplicates(["product_id"])

In [0]:
cart_items_enriched_df = cart_items_df.join(
    product_master_df,
    on="product_id",
    how="left"
)


## Data Read

### Load Silver Table
Reads cart items data from Silver layer.

### Output
Displays the dataset for analysis.

In [0]:
df = spark.read.table("ecom_catalog.silver.cart_items")
display(df)

In [0]:
df = spark.table("ecom_catalog.silver.cart_items")
display(df.limit(20))

product_id,cart_id,product_name,price,quantity,product_total,product_discounted_total,discount_percentage,thumbnail,revenue,discount_value,category,brand
124,2,iPhone X,899.99,4,3599.96,3310.88,8.03,https://cdn.dummyjson.com/products/images/smartphones/iPhone%20X/thumbnail.png,3599.96,289.0799999999999,null,null
112,6,TV Studio Camera Pedestal,499.99,3,1499.97,1264.62,15.69,https://cdn.dummyjson.com/products/images/mobile-accessories/TV%20Studio%20Camera%20Pedestal/thumbnail.png,1499.97,235.35000000000014,null,null
178,9,Corset Leather With Skirt,89.99,2,179.98,157.32,12.59,https://cdn.dummyjson.com/products/images/womens-dresses/Corset%20Leather%20With%20Skirt/thumbnail.png,179.98,22.659999999999997,null,null
111,15,Selfie Stick Monopod,12.99,3,38.97,34.69,10.98,https://cdn.dummyjson.com/products/images/mobile-accessories/Selfie%20Stick%20Monopod/thumbnail.png,38.97,4.280000000000001,null,null
74,17,Spoon,4.99,4,19.96,19.41,2.78,https://cdn.dummyjson.com/products/images/kitchen-accessories/Spoon/thumbnail.png,19.96,0.5500000000000007,null,null
90,20,Puma Future Rider Trainers,89.99,5,449.95,383.81,14.7,https://cdn.dummyjson.com/products/images/mens-shoes/Puma%20Future%20Rider%20Trainers/thumbnail.png,449.95,66.13999999999999,null,null
83,27,Blue & Black Check Shirt,29.99,5,149.95,136.81,8.76,https://cdn.dummyjson.com/products/images/mens-shirts/Blue%20&%20Black%20Check%20Shirt/thumbnail.png,149.95,13.139999999999986,null,null
153,29,Volleyball,11.99,5,59.95,50.33,16.05,https://cdn.dummyjson.com/products/images/sports-accessories/Volleyball/thumbnail.png,59.95,9.620000000000005,null,null
46,30,Plant Pot,14.99,4,59.96,49.38,17.65,https://cdn.dummyjson.com/products/images/home-decoration/Plant%20Pot/thumbnail.png,59.96,10.579999999999998,null,null
168,1,Charger SXT RWD,32999.99,3,98999.97,85743.87,13.39,https://cdn.dummyjson.com/products/images/vehicle/Charger%20SXT%20RWD/thumbnail.png,98999.97,13256.100000000006,null,null
